# Project Scaffold — databricks-lakehouse-mastery

Run all cells to create the project directory tree and starter stub files.

```
databricks-lakehouse-mastery/
├── 00_workspace_sanity.py
├── 01_batch_processing/
│   ├── 01_historical_api_ingest.py
│   └── 02_schema_transformation.py
├── 02_delta_lakehouse/
│   ├── 01_acid_merge_upsert.py
│   └── 02_time_travel_compaction.py
├── 03_structured_streaming/
│   ├── 01_stream_landing_producer.py
│   └── 02_stream_to_delta_bronze.py
├── 04_stream_aggregations/
│   └── 01_watermarking_sliding_windows.py
├── 05_declarative_dlt/
│   └── 01_dlt_medallion_pipeline.py
├── 06_orchestration_debugging/
│   └── master_pipeline_runner.py
└── README.md
```

Safe to re-run: existing files are skipped, nothing is overwritten or deleted.

In [0]:
# ---------------------------------------------------------------------------
# Project configuration: base path and starter file templates
# ---------------------------------------------------------------------------

BASE = "/Workspace/Users/raghukonda17@gmail.com/databricks-lakehouse-mastery"


def py_stub(title: str, purpose: str) -> str:
    """Starter template for a project module."""
    return f'''"""
{title}

{purpose}
"""

# TODO: implement this module
'''


FILES = {
    "00_workspace_sanity.py": py_stub(
        "00 - Workspace Sanity Check",
        "Verify compute access, Unity Catalog permissions, and required libraries before starting the course modules.",
    ),
    "01_batch_processing/01_historical_api_ingest.py": py_stub(
        "01 - Historical API Ingest (Batch)",
        "Pull historical data from a public REST API in batch and land it as raw files in a Unity Catalog volume.",
    ),
    "01_batch_processing/02_schema_transformation.py": py_stub(
        "02 - Schema Transformation",
        "Apply an explicit schema and transform raw batch data into a clean, typed DataFrame for the silver layer.",
    ),
    "02_delta_lakehouse/01_acid_merge_upsert.py": py_stub(
        "01 - ACID Merge / Upsert",
        "Demonstrate MERGE-based upserts on Delta tables for idempotent incremental loads.",
    ),
    "02_delta_lakehouse/02_time_travel_compaction.py": py_stub(
        "02 - Time Travel & Compaction",
        "Explore Delta time travel (VERSION AS OF, DESCRIBE HISTORY) and table maintenance (OPTIMIZE, VACUUM).",
    ),
    "03_structured_streaming/01_stream_landing_producer.py": py_stub(
        "01 - Stream Landing Producer",
        "Producer that writes simulated event data to the landing location consumed by the streaming job.",
    ),
    "03_structured_streaming/02_stream_to_delta_bronze.py": py_stub(
        "02 - Stream to Delta Bronze",
        "Structured Streaming job: read new files from the landing location and append them to the bronze Delta table.",
    ),
    "04_stream_aggregations/01_watermarking_sliding_windows.py": py_stub(
        "01 - Watermarking & Sliding Windows",
        "Stateful stream aggregations using watermarking and sliding windows, written to a Delta sink.",
    ),
    "05_declarative_dlt/01_dlt_medallion_pipeline.py": py_stub(
        "01 - SDP Medallion Pipeline",
        "Lakeflow Spark Declarative Pipeline implementing the medallion architecture (bronze to silver to gold) with data quality expectations.",
    ),
    "06_orchestration_debugging/master_pipeline_runner.py": py_stub(
        "Master Pipeline Runner",
        "Orchestrate all modules end-to-end as a Lakeflow Job and demonstrate debugging techniques.",
    ),
    "README.md": (
        "# databricks-lakehouse-mastery\n"
        "\n"
        "Hands-on Databricks Lakehouse learning project.\n"
        "\n"
        "## Module map\n"
        "- `00_workspace_sanity.py` — verify compute, catalogs, and libraries\n"
        "- `01_batch_processing/` — batch ingest + schema transformation\n"
        "- `02_delta_lakehouse/` — ACID merge/upsert, time travel, compaction\n"
        "- `03_structured_streaming/` — landing producer + stream-to-Delta bronze\n"
        "- `04_stream_aggregations/` — watermarking & sliding windows\n"
        "- `05_declarative_dlt/` — SDP medallion pipeline\n"
        "- `06_orchestration_debugging/` — master pipeline runner (Lakeflow Jobs)\n"
        "\n"
        "## Getting started\n"
        "Run `00_workspace_sanity.py` first to verify your environment.\n"
    ),
}

print(f"Configured {len(FILES)} files under {BASE}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceDoesNotExist
from databricks.sdk.service.workspace import ImportFormat

w = WorkspaceClient()

# 1) Create all parent directories (idempotent)
directories = sorted({p.rsplit("/", 1)[0] for p in FILES if "/" in p})
for d in directories:
    w.workspace.mkdirs(f"{BASE}/{d}")
    print(f"[dir ] {d}/")

# 2) Create each file only if it does not already exist (safe to re-run)
for rel_path, content in FILES.items():
    full_path = f"{BASE}/{rel_path}"
    try:
        w.workspace.get_status(full_path)
        print(f"[skip] {rel_path} (already exists)")
    except ResourceDoesNotExist:
        w.workspace.upload(full_path, content.encode("utf-8"), format=ImportFormat.SOURCE)
        print(f"[file] {rel_path}")

print(f"\nDone: {len(directories)} directories and {len(FILES)} files configured in {BASE}")

In [0]:
def print_tree(path: str, prefix: str = ""):
    """Recursively print the workspace tree under `path`."""
    for obj in sorted(w.workspace.list(path), key=lambda o: o.path):
        name = obj.path.rsplit("/", 1)[-1]
        if obj.object_type.value == "DIRECTORY":
            print(f"{prefix}{name}/")
            print_tree(obj.path, prefix + "    ")
        else:
            print(f"{prefix}{name}")


print("databricks-lakehouse-mastery/")
print_tree(BASE)